In [1]:
import os, sys, json, re, requests
project_root = r"C:\Users\User\pg-screening-system"
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocessing.segmenter import segment_document
from src.extractors.gpa_extractor import extract_gpa
print("Setup OK")

Setup OK


In [4]:
import fitz
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def extract_text_ocr(pdf_path: str) -> dict:
    doc  = fitz.open(pdf_path)
    text = "".join(p.get_text() for p in doc)

    if len(text.split()) >= 80:
        return {"text": text, "ocr_used": False, "confidence": 1.0}

    images = convert_from_path(pdf_path, dpi=300)
    full_text, all_confs = "", []

    for img in images:
        gray = img.convert("L")
        data = pytesseract.image_to_data(
            gray, output_type=pytesseract.Output.DICT)
        confs = [c for c in data["conf"] if c != -1]
        all_confs.extend(confs)
        full_text += pytesseract.image_to_string(gray) + "\n"

    conf = round(sum(all_confs) / len(all_confs) / 100, 3) if all_confs else 0.0

    return {
        "text":       full_text,
        "ocr_used":   True,
        "confidence": conf,
        "pages":      len(images)
    }

result = extract_text_ocr("data/raw/transcripts/225127909 T.pdf")

print(f"OCR utilisé  : {result['ocr_used']}")
print(f"Confiance    : {result['confidence']}")
print(f"Mots extraits: {len(result['text'].split())}")
print()
print("--- TEXTE BRUT OCR ---")
print(result['text'][:800])

OCR utilisé  : True
Confiance    : 0.68
Mots extraits: 349

--- TEXTE BRUT OCR ---
Southeast Missouri State University Record of: Devon Daniel Mark Serra

Cape Girardeau, Missouri Social Security Number: ***-**- 6357
Date of Birth: 13-JAN-1998

Level: Graduate Master

Official Transcript Date Issued: 46.AUG.2024

Document is printed on red Security paper With Visible aid MUOFESCEnE bers Hid Woes Hol Fequire a raised seal, Valid only when it hears the signature of the
Registrar, a Criss-Cross watermark viewed Mont both Sides OF The paper thd Wei Bey HOX appears at he lower right Corner, Gray box will disappear and
SValid) willappear when rubbed With Fingers TOMATO PONT THIS PeCORU TTUSE HOt bE Feleised (0 i ThiFd party Without written permission from the student.

Degree Awarded: Master of Science 15-MAY-2021
Major : Applied Computer Science (MS)

SUBJ NO. COURSE TITLE CR


In [5]:
gpa_result = extract_gpa(result['text'], degree_level="bsc")

print("=== RÉSULTAT REGEX GPA ===")
print(f"GPA trouvé   : {gpa_result.raw_value}")
print(f"Échelle      : {gpa_result.original_scale}")
print(f"Normalisé    : {gpa_result.normalised_gpa}")
print(f"Confiance    : {gpa_result.confidence}")
print(f"Source       : {gpa_result.source}")
print(f"Match brut   : {gpa_result.raw_match}")
print()
if gpa_result.confidence < 0.70:
    print("⚠️  Confiance < 0.70 → LLM fallback déclenché")
else:
    print("✅ Regex suffisant — LLM non nécessaire")

=== RÉSULTAT REGEX GPA ===
GPA trouvé   : 4.0
Échelle      : 4.0
Normalisé    : 4.0
Confiance    : 0.75
Source       : implicit_4
Match brut   : GPA: 4.00

✅ Regex suffisant — LLM non nécessaire


In [14]:
def call_ollama_gpa_v2(text: str, timeout: int = 120) -> dict:
    prompt = (
        "You are an academic transcript parser.\n"
        "Find the FINAL CUMULATIVE GPA in the transcript.\n\n"
        "IMPORTANT RULES:\n"
        "- GPA scale is almost always 4.0 unless explicitly stated otherwise\n"
        "- Numbers like 114, 30, 45 are CREDITS not GPA scale — ignore them\n"
        "- Look for patterns like: GPA: 3.80  or  CGPA: 3.5/4.0  or  Grade: 16/20\n"
        "- If you see 'GPA: 3.80' with no scale mentioned → scale = 4.0\n\n"
        "Return ONLY valid JSON:\n"
        '{"gpa_raw": <number>, '
        '"gpa_scale": <4.0 unless explicitly different>, '
        '"gpa_normalized_4": <gpa_raw if scale=4.0, else convert>, '
        '"confidence": <0.0-1.0>, '
        '"evidence": "<exact line where you found GPA>"}\n\n'
        f"Transcript:\n{text[:2000]}\n\nJSON:"
    )
    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "mistral:7b-instruct",
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.1}
            },
            timeout=timeout
        )
        raw = r.json().get("response", "").strip()
        print("Réponse brute LLM v2 :")
        print(raw[:400])
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {}
    except Exception as e:
        print(f"Erreur : {e}")
        return {}

print("Appel LLM...")
llm_gpa_v2 = call_ollama_gpa_v2(result['text'])

print("\n=== RÉSULTAT LLM v2 ===")
print(json.dumps(llm_gpa_v2, indent=2))

Appel LLM...
Réponse brute LLM v2 :
{
"gpa_raw": 3.80,
"gpa_scale": 4.0,
"gpa_normalized_4": 3.80,
"confidence": 1.0,
"evidence": "TOTAL INSTITUTION 30.00 30,00. 114.0 3.80"
}

=== RÉSULTAT LLM v2 ===
{
  "gpa_raw": 3.8,
  "gpa_scale": 4.0,
  "gpa_normalized_4": 3.8,
  "confidence": 1.0,
  "evidence": "TOTAL INSTITUTION 30.00 30,00. 114.0 3.80"
}


In [15]:
print("=== BILAN TEST TRANSCRIPT SCANNÉ ===")
print()
print(f"OCR confidence     : {result['confidence']}")
print(f"Regex GPA          : {gpa_regex.raw_value} (conf={gpa_regex.confidence})")
print(f"LLM v1 GPA         : 3.80 scale=114 ❌ (erreur crédits)")
print(f"LLM v2 GPA         : 3.80 scale=4.0 ✅ (prompt corrigé)")
print()
print("--- DÉCISION PIPELINE ---")
print("✅ GPA final : 3.80 / 4.0")
print("   Source    : llm (prompt v2)")
print("   Confiance : 1.0")
print("   needs_review : False")
print()
print("=== LEÇONS APPRISES ===")
print("1. Prompt engineering critique — règle 'scale > 10 = crédits' nécessaire")
print("2. Validation post-LLM obligatoire — détecter incohérences gpa_raw vs scale")
print("3. Evidence utile — LLM cite exactement la ligne source")

=== BILAN TEST TRANSCRIPT SCANNÉ ===

OCR confidence     : 0.68


NameError: name 'gpa_regex' is not defined